In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'd:\\Tipto\\agentic-ai-projects\\hr-policy-agent'

In [3]:
import pymupdf
doc = pymupdf.open("data/raw/Human_Resource_Policy_Manual_PRAAN.2020.pdf")
len(doc)

64

In [6]:
from pathlib import Path
page_save_dir = Path("data/pages")
page_save_dir.mkdir(parents=True, exist_ok=True) 

In [7]:
for p in range(doc.page_count):
    page = doc[p]
    pix = page.get_pixmap(dpi = 300)
    pix.save(f"{page_save_dir}/page_{p + 1:03d}.png")

In [35]:
def skew_angle(ink, lo=-3, hi=3, step=0.25):
    """Return rotation angle (degrees) that best straightens text lines."""
    small = cv2.resize(ink, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    h, w = small.shape
    best, best_score = 0.0, -1.0
    for a in np.arange(lo, hi + 1e-6, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1)
        r = cv2.warpAffine(small, M, (w, h), flags=cv2.INTER_NEAREST)
        score = np.var(r.sum(axis=1))
        if score > best_score:
            best_score, best = score, float(a)
    return best

In [36]:
def preprocess(bgr):
    """Denoise then deskew a page image."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, h=8)
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    a = skew_angle(ink)
    h, w = bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
    return cv2.warpAffine(bgr, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderValue=(255, 255, 255)), a

In [37]:
# ocr with layout boxes
from rapidocr_onnxruntime import RapidOCR
rapid = RapidOCR()

def ocr_page(bgr):
    """Run RapidOCR on an image; return list of {box, text, score}."""
    result, _ = rapid(bgr)
    return [
        {"box": [[float(x), float(y)] for x, y in b],
         "text": t, "score": float(s)}
        for b, t, s in (result or [])
    ]

In [38]:
def to_lines(items, y_tol_factor=0.6):
    """Cluster OCR items into lines by y-center proximity."""
    if not items:
        return []
    heights = [it["box"][2][1] - it["box"][0][1] for it in items]
    y_tol = y_tol_factor * float(np.median(heights))

    items = sorted(items, key=lambda it: np.mean([p[1] for p in it["box"]]))
    lines, cur, cur_y = [], [], None
    for it in items:
        y = float(np.mean([p[1] for p in it["box"]]))
        if cur_y is None or abs(y - cur_y) <= y_tol:
            cur.append(it)
            cur_y = y if cur_y is None else (cur_y + y) / 2
        else:
            lines.append(cur)
            cur, cur_y = [it], y
    if cur:
        lines.append(cur)
    # sort each line left → right
    return [sorted(l, key=lambda it: it["box"][0][0]) for l in lines]

In [74]:
def collapse_consecutive(idx, gap=3):
    """Collapse adjacent indices into single boundaries."""
    if len(idx) == 0:
        return idx
    
    out, start = [idx[0]], idx[0]
    
    for x in idx[1:]:
        if x - start > gap:
            out.append(start)
            out.append(x)
        
        start = x
    
    out.append(idx[-1])
    return np.array(out)

In [75]:
def find_table_grid(bw, items=None, min_rows=3, min_cols=3, min_cell_ratio=0.6):
    """
    Detect a real table grid.
    Returns (rows, cols) or ([], []) if no plausible table is found.

    Guards applied:
      - need >= min_rows horizontal lines AND >= min_cols vertical lines
      - the detected grid must cover enough of the page (< 90% is suspicious
        for a full-page table, but > 5% to avoid tiny decorative boxes)
      - if `items` (OCR boxes) is provided, at least `min_cell_ratio`
        of cells must contain some OCR text.
    """
    h, w = bw.shape
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, w // 12), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, h // 12)))
    inv = 255 - bw
    hl = cv2.morphologyEx(inv, cv2.MORPH_OPEN, hk)
    vl = cv2.morphologyEx(inv, cv2.MORPH_OPEN, vk)

    rows = np.where(hl.sum(axis=1) > 0.4 * 255 * w)[0]
    cols = np.where(vl.sum(axis=0) > 0.4 * 255 * h)[0]
    rows = collapse_consecutive(rows)
    cols = collapse_consecutive(cols)

    # Basic shape check
    if len(rows) < min_rows or len(cols) < min_cols:
        return np.array([]), np.array([])

    # Cell size sanity: cells should be at least ~15 px tall/wide
    row_gaps = np.diff(rows)
    col_gaps = np.diff(cols)
    if row_gaps.min() < 15 or col_gaps.min() < 15:
        return np.array([]), np.array([])

    # Content check: at least `min_cell_ratio` of cells must contain OCR text
    if items is not None and len(items) > 0:
        n_r, n_c = len(rows) - 1, len(cols) - 1
        filled = 0
        for r in range(n_r):
            for c in range(n_c):
                y0, y1 = rows[r], rows[r + 1]
                x0, x1 = cols[c], cols[c + 1]
                # does any OCR box center fall inside this cell?
                for it in items:
                    cx = (it["box"][0][0] + it["box"][2][0]) / 2
                    cy = (it["box"][0][1] + it["box"][2][1]) / 2
                    if x0 <= cx <= x1 and y0 <= cy <= y1:
                        filled += 1
                        break
        ratio = filled / (n_r * n_c)
        if ratio < min_cell_ratio:
            return np.array([]), np.array([])

    return rows, cols

In [42]:
import re
H1 = re.compile(r"^Section\s+\d+\b", re.I)
H2 = re.compile(r"^\d+\.\d+\s+\S")
H3 = re.compile(r"^\d+\.\d+\.\d+\s+\S")
LI = re.compile(r"^(?:[a-z]\.|\([ivx]+\)|[-•])\s", re.I)
TRAILING_PAGE = re.compile(r"\s+\d{1,3}\s*$")


def strip_trailing_page(t: str) -> str:
    """Only strip a trailing page number if the line looks like a heading/list entry."""
    if H1.match(t) or H2.match(t) or H3.match(t) or LI.match(t):
        return TRAILING_PAGE.sub("", t).strip()
    return t.strip()


def classify_block(text):
    t = strip_trailing_page(text.strip())
    if H3.match(t): return "h3"
    if H2.match(t): return "h2"
    if H1.match(t): return "h1"
    if LI.match(t): return "list"
    if t.isupper() and 4 <= len(t) <= 80: return "h3"
    return "para"

In [41]:
def ocr_cells(bgr, rows, cols):
    """OCR each cell of a detected grid independently (with 2x zoom)."""
    table = []
    for r in range(len(rows) - 1):
        row = []
        for c in range(len(cols) - 1):
            crop = bgr[rows[r]:rows[r + 1], cols[c]:cols[c + 1]]
            if crop.size == 0 or crop.shape[0] < 8 or crop.shape[1] < 8:
                row.append("")
                continue
            crop = cv2.resize(crop, None, fx=2, fy=2,
                              interpolation=cv2.INTER_CUBIC)
            res, _ = rapid(crop)
            row.append(" ".join(t for _, t, _ in (res or [])))
        table.append(row)
    return table

In [54]:
from collections import Counter
VOCAB = Counter()

def add_to_vocab(text: str):
    """Feed a cleaned line back so future splits can use these words."""
    for w in re.findall(r"[A-Za-z]{2,}", text):
        VOCAB[w.lower()] += 1

In [55]:
SUFFIXES = ("org", "com", "net", "gov", "edu", "bd", "info")

In [56]:
def split_run(tok: str) -> str:
    """Insert spaces at case / digit boundaries:
    'DailySubsistenceAllowance' -> 'Daily Subsistence Allowance'."""
    if not tok or len(tok) < 4:
        return tok
    if any(c in tok for c in ".,@/|:"):
        return tok                        # emails, urls, dates, paths
    parts = re.split(r"(?<=[a-z])(?=[A-Z])", tok)
    parts = [p for grp in parts
             for p in re.split(r"(?<=\d)(?=[A-Za-z])|(?<=[A-Za-z])(?=\d)", grp)
             if p]
    return " ".join(parts) if len(parts) > 1 else tok

In [66]:
STATIC_WORDS = {
    # short connector words
    "a","an","and","as","at","be","by","for","from","if","in","into",
    "is","it","no","not","of","on","or","the","to","up","via","with",
    # HR / policy specific
    "human","resource","policy","manual","development","process","purpose",
    "application","revision","interpretation","professional","code","conduct",
    "implementation","monitoring","service","rules","title","commitment",
    "definitions","organizational","structure","management","position",
    "classification","grade","general","principles","function","team",
    "gender","committee","focal","point","employment","about","index",
    "contents","page","acronyms","curriculum","vitae","manager","deputy",
    "chief","executive","officer","director","government","bangladesh",
    "information","system","description","evaluation","accountability",
    "learning","affairs","bureau","advocacy","mass","communication",
    "project","proposal","research","documentation","authority","terms",
    "reference","network","participatory","action","adopted","adopting",
    "approved","approving","reviewed","reviewing","committee","chairperson",
    "allowance","subsistence","remuneration","gratuity","provident",
    "maternity","paternity","disciplinary","grievance","supervisor",
    "employee","employees","probation","redundancy","termination",
    "resignation","settlement","compensation","salary","salaries","benefit",
    "benefits","leave","travel","accommodation","perdiem","diem","notice",
    "termination","resignation","conflict","settlement","appeal","suspension",
    "enquiry","investigation","dismissal","misconduct","warning","hearing",
    # months
    "january","february","march","april","may","june","july",
    "august","september","october","november","december",
    # generic English words common in this doc
    "list","total","tota1","position","salary","step","steps","grade",
    "grades","basic","gross","net","amount","payable","paid","deduction",
    "deductions","fund","year","years","month","months","day","days",
    "week","weeks","hour","hours","time","times","date","dates",
    "full","part","short","long","work","working","worker","workers",
    "staff","member","members","section","sections","clause","clauses",
    "rule","rules","paragraph","paragraphs","title","heading",
    "approved","approving","reviewed","reviewing","signed","signing",
    "chairperson","chief","executive","director","department",
}

In [87]:
def _known(w: str) -> bool:
    w = w.lower()
    return w in STATIC_WORDS or VOCAB[w] >= 1

def split_lowercase_runs(tok: str) -> str:
    """
    Split a merged alphabetic token into known words.
    Only splits when ALL resulting pieces are known words.
    Never splits proper nouns (Sharmind, Neelormi, Masud).
    """
    if not tok.isalpha() or len(tok) < 6:
        return tok

    # Proper noun guard — capitalized, rest lowercase, 4–15 chars
    if tok[:1].isupper() and tok[1:].islower() and 4 <= len(tok) <= 15:
        # still allow split if a known multi-word exists with this casing,
        # but for name-shaped tokens we bail
        return tok

    # If the whole token is already known, don't split
    if _known(tok):
        return tok

    n = len(tok)
    INF = 10**9
    dp = [(INF, -1)] * (n + 1)
    dp[0] = (0, -1)

    for i in range(1, n + 1):
        for j in range(max(0, i - 15), i):
            w = tok[j:i]
            if _known(w):
                cost = 0
            else:
                # unknown piece: only allowed as first or last, len >= 3
                if j == 0 or i == n:
                    if len(w) < 3:
                        continue
                    cost = 5          # cheap-ish unknown prefix/suffix
                else:
                    continue          # unknown middle piece → not allowed
            total = dp[j][0] + cost
            if total < dp[i][0]:
                dp[i] = (total, j)

    if dp[n][0] >= INF:
        return tok    # no valid split found → keep whole token

    pieces, k = [], n
    while k > 0:
        _, j = dp[k]
        pieces.append(tok[j:k])
        k = j
    pieces.reverse()

    # Sanity: reject splits with any piece < 3 chars that isn't a real word
    if any(len(p) < 3 and not _known(p) for p in pieces):
        return tok
    if len(pieces) < 2:
        return tok

    return " ".join(pieces)

In [59]:
def split_suffix(tok: str) -> str:
    """'developmentorg.' -> 'development org.'
    Safe: skips tokens containing @ | : / or multiple dots."""
    if not tok or len(tok) < 6:
        return tok
    if any(c in tok for c in "@|:/"):
        return tok
    # Separate trailing punctuation
    m = re.match(r"^([A-Za-z]+)([.,;:!?\)\]]*)$", tok)
    if not m:
        return tok
    core, trail = m.group(1), m.group(2)
    low = core.lower()
    for s in SUFFIXES:
        if low.endswith(s) and len(core) > len(s) + 1:
            return core[:-len(s)] + " " + core[-len(s):] + trail
    return tok

In [88]:
def respace_line(text: str) -> str:
    out = []
    for tok in text.split():
        # split on comma/slash but keep the delimiter
        parts = re.split(r"([,/])", tok)
        fixed = []
        for part in parts:
            if part in (",", "/"):
                fixed.append(part)
                continue
            part = split_run(part)
            sub = []
            for t in part.split():
                t = split_suffix(t)
                t = split_lowercase_runs(t)
                sub.append(t)
            fixed.append(" ".join(sub))
        out.append("".join(fixed))
    return " ".join(out)

In [80]:
STOP_WORDS = {"of", "and", "the", "a", "an", "in", "on", "at", "to", "for",
              "by", "with", "from", "or"}

def titlecase_heading(s):
    words = s.split()
    out = []
    for i, w in enumerate(words):
        # keep all-caps words as-is (acronyms)
        if w.isupper() and len(w) > 1:
            out.append(w)
        elif i == 0 or i == len(words) - 1:
            out.append(w[:1].upper() + w[1:])
        elif w.lower() in STOP_WORDS:
            out.append(w.lower())
        else:
            out.append(w[:1].upper() + w[1:])
    return " ".join(out)

In [61]:
HR_DICT = {
    "committee","maternity","paternity","remuneration","gratuity",
    "provident","disciplinary","grievance","supervisor","employee",
    "employees","probation","allowance","subsistence","advocacy",
    "network","participatory","acronyms","chairperson","redundancy",
    "termination","resignation","settlement","compensation",
    "curriculum","vitae","executive","approved","reviewed",
    "january","february","march","april","may","june","july",
    "august","september","october","november","december",
}

In [62]:
def autocorrect_word(tok: str) -> str:
    low = tok.lower()
    if low in HR_DICT or not tok.isalpha() or len(tok) < 5:
        return tok
    m = difflib.get_close_matches(low, HR_DICT, n=1, cutoff=0.85)
    if m:
        c = m[0]
        return c.capitalize() if tok[0].isupper() else c
    return tok

In [63]:
MONTH_FIXES = [
    (re.compile(r"\bDe\s+er\s*(\d{4})", re.I), r"December \1"),
    (re.compile(r"\bNov\s+em\s*ber\s*(\d{4})", re.I), r"November \1"),
    (re.compile(r"\bOct\s+o\s*ber\s*(\d{4})", re.I), r"October \1"),
]

def fix_months(s: str) -> str:
    for pat, rep in MONTH_FIXES:
        s = pat.sub(rep, s)
    return s

In [64]:
def clean_line(text: str) -> str:
    text = respace_line(text)
    text = fix_months(text)
    text = " ".join(autocorrect_word(w) for w in text.split())
    return text

## Assemble Final Document

In [34]:
import pymupdf
import numpy as np
import cv2

def render_page(doc, page_idx, dpi=300):
    """
    Render one PDF page to an OpenCV BGR numpy array.
    Args:
        doc:      pymupdf.Document (already opened)
        page_idx: 0-based page index
        dpi:      rasterization DPI
    Returns:
        bgr: np.ndarray (H, W, 3) uint8
    """
    page = doc[page_idx]
    pix = page.get_pixmap(dpi=dpi, alpha=False)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
        pix.height, pix.width, pix.n
    )
    if pix.n == 3:
        bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    elif pix.n == 4:
        bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    else:
        bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return np.ascontiguousarray(bgr)

In [81]:
def render_markdown_table(rows):
    """Rows: list of lists of strings. GitHub-flavoured Markdown table."""
    if not rows:
        return ""
    ncol = max(len(r) for r in rows)
    rows = [r + [""] * (ncol - len(r)) for r in rows]

    def cell(s):
        return str(s).replace("|", "\\|").replace("\n", " ").strip()

    header, body = rows[0], rows[1:]
    lines = ["| " + " | ".join(cell(c) for c in header) + " |"]
    lines.append("|" + "|".join(["---"] * ncol) + "|")
    for r in body:
        lines.append("| " + " | ".join(cell(c) for c in r) + " |")
    return "\n".join(lines)


def render_markdown(blocks):
    """Convert structured blocks into Markdown with page markers."""
    out_lines, current_page = [], None
    for b in blocks:
        if b.get("page") != current_page:
            current_page = b["page"]
            out_lines.append(f"\n<!-- page {current_page} -->\n")

        t = b.get("type", "para")
        if t == "table":
            out_lines.append(render_markdown_table(b["data"]))
            out_lines.append("")
        elif t == "h1":
            out_lines.append(f"# {titlecase_heading(b['text'].strip())}\n")
        elif t == "h2":
            out_lines.append(f"## {titlecase_heading(b['text'].strip())}\n")
        elif t == "h3":
            out_lines.append(f"### {titlecase_heading(b['text'].strip())}\n")
        elif t == "list":
            out_lines.append(b["text"].strip())
        else:
            out_lines.append(b["text"].strip() + "\n")

    md = "\n".join(out_lines)
    md = re.sub(r"\n{3,}", "\n\n", md)
    return md.strip() + "\n"

In [82]:
def strip_boilerplate(blocks, min_frac=0.5):
    from collections import Counter
    # normalize: strip digits, keep letter skeleton
    def skeleton(t):
        return re.sub(r"\d+", "#", t.lower())[:80]
    counts = Counter(skeleton(b["text"]) for b in blocks if b["type"] == "para")
    n_pages = len({b["page"] for b in blocks})
    boiler = {k for k, c in counts.items() if c >= min_frac * n_pages}
    return [b for b in blocks
            if b["type"] != "para" or skeleton(b["text"]) not in boiler]

In [89]:
def clean_line(text):
    text = respace_line(text)
    text = fix_months(text)
    text = " ".join(autocorrect_word(w) for w in text.split())
    return text

In [83]:
import json, time
DPI = 300

def build_document(pdf_path, out_dir, max_pages=None):
    doc = pymupdf.open(pdf_path)
    n = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
    blocks = []

    for p in range(n):
        t0 = time.time()
        bgr = render_page(doc, p, dpi=DPI)
        bgr, angle = preprocess(bgr)
        items = ocr_page(bgr)
        lines = to_lines(items)

        bw = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

        rows, cols = find_table_grid(bw, items=items)

        if len(rows) >= 3 and len(cols) >= 3:
            table = ocr_cells(bgr, rows, cols)
            blocks.append({
                "page": p + 1,
                "type": "table",
                "data": table,
                "grid": {"rows": rows.tolist(), "cols": cols.tolist()},
            })
        else:
            # paragraph path — one line at a time
            for line in lines:
                raw = " ".join(it["text"] for it in line)
                text = clean_line(raw)
                if not text.strip():
                    continue
                add_to_vocab(text)                       # feed vocab (if using)
                blocks.append({
                    "page": p + 1,
                    "type": classify_block(text),
                    "text": text,
                    "bbox": line[0]["box"][0] + line[-1]["box"][2],
                })

        print(f"page {p+1:3d}/{n}  angle={angle:+.2f}  "
              f"blocks+={len(blocks)}  {time.time()-t0:.1f}s")
    
    blocks = strip_boilerplate(blocks, min_frac=0.5)
    md = render_markdown(blocks)
    (out_dir / "hr_manual.md").write_text(md, encoding="utf-8")
    (out_dir / "hr_manual.json").write_text(
        json.dumps({"blocks": blocks, "source": Path(pdf_path).name},
                   indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return md, blocks

In [22]:
def ocr_page_words(bgr):
    """OCR a page and return items.
    Each item's box is a word-level box.
    """
    result, _ = rapid(bgr)
    items = []
    for box, text, score in (result or []):
        # RapidOCR boxs usually enclose a whole line, split on whitespace runs by re-detecting gaps between characters
        items.append(
            {
                "box": [[float(x), float(y)] for x, y in box],
                "text": text,
                "score": float(score),
            }
        )

In [23]:
import re
from functools import lru_cache

# Build once from the whole document's vocabulary
WORD_VOCAB = set()   # lowercase words seen with spaces around them in clean pages

In [90]:
CAMEL = re.compile(r"[a-z][A-Z]")  # boundary between lower -> upper
LOWER_UPPER = re.compile(r"(?<=[a-z])(?=[A-Z])")

def split_run(tok: str) -> str:
    """
    Best-effort reinsertion of spaces into a merged token.
    'DailySubsistenceAllowance' → 'Daily Subsistence Allowance'
    'ParticipatoryResearchActionNetwork' → 'Participatory Research Action Network'
    'pranbd.org' → unchanged (has a dot)
    """
    if not tok or len(tok) < 4:
        return tok
    if "." in tok or "," in tok or "@" in tok:
        return tok  # emails, urls, numbers — leave alone
    # split at lower -> Upper boundaries
    parts = re.split(r"(?<=[a-z])(?=[A-Z])", tok)
    # also split at digit↔letter boundary inside a word, but not before '.' etc.
    parts = [p for grp in parts for p in re.split(r"(?<=\d)(?=[A-Za-z])|(?<=[A-Za-z])(?=\d)", grp) if p]
    return " ".join(parts) if len(parts) > 1 else tok


def respace_line(text: str) -> str:
    # split on whitespace to preserve existing spaces, then respace each token
    return " ".join(split_run(tok) for tok in text.split())

In [25]:
SUFFIXES = ("org", "com", "net", "gov", "edu", "bd", "org.", "com.")
def split_suffix(tok):
    for s in SUFFIXES:
        if tok.endswith(s) and len(tok) > len(s):
            return tok[:-len(s)] + " " + tok[-len(s):]
    return tok

In [26]:
def render_page_adaptive(doc, page_idx, base_dpi=300):
    page = doc[page_idx]
    # measure the smallest font on the page from the embedded invisible layer
    # (yes, it's low-quality, but it's fine for *sizing*)
    spans = page.get_text("dict")["blocks"]
    sizes = [s["size"] for b in spans if b.get("type") == 0
             for l in b.get("lines", []) for s in l.get("spans", [])]
    min_pt = min(sizes) if sizes else 12
    dpi = 400 if min_pt < 8 else base_dpi
    return render_page(doc, page_idx, dpi=dpi), min_pt

In [27]:
import difflib

HR_DICT = {
    "committee", "maternity", "paternity", "remuneration", "gratuity",
    "provident", "disciplinary", "grievance", "supervisor", "employee",
    "employees", "probation", "allowance", "subsistence", "advocacy",
    "network", "participatory", "acronyms", "chairperson", "redundancy",
    "termination", "resignation", "settlement", "compensation",
    "curriculum", "vitae", "executive", "approved", "reviewed",
    "december", "november", "october", "september", "august",
}

def autocorrect_word(tok: str) -> str:
    low = tok.lower()
    if low in HR_DICT:
        return tok
    # only correct alphabetic tokens of reasonable length
    if not tok.isalpha() or len(tok) < 5:
        return tok
    m = difflib.get_close_matches(low, HR_DICT, n=1, cutoff=0.85)
    if m:
        corrected = m[0]
        return corrected.capitalize() if tok[0].isupper() else corrected
    return tok

In [91]:
def clean_line(text: str) -> str:
    text = respace_line(text)
    text = fix_months(text)
    out = []
    for raw in text.split():
        # Split token into leading punctuation + alphabetic core + trailing punctuation
        m = re.match(r"^([^\w]*)([A-Za-z]+)([^\w]*)$", raw)
        if m:
            pre, core, post = m.groups()
            out.append(pre + autocorrect_word(core) + post)
        else:
            # Token has digits or other characters mixed in — split on letter runs
            parts = re.split(r"([A-Za-z]+)", raw)
            cleaned = []
            for p in parts:
                if p.isalpha() and len(p) >= 5:
                    cleaned.append(autocorrect_word(p))
                else:
                    cleaned.append(p)
            out.append("".join(cleaned))
    return " ".join(out)

In [70]:
HR_DICT |= {
    "committee", "chairperson", "executive", "adopted", "approved",
    "reviewed", "organization", "participatory", "development",
    "management", "documentation", "information", "accountability",
}

In [29]:
MONTH_RE = re.compile(
    r"\b(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|June?|July?|"
    r"Aug(?:ust)?|Sep(?:t|tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\b",
    re.I,
)
def fix_months(s):
    # 'De er2021' → the 'De' matches Dec prefix, then 'er2021' follows
    s = re.sub(r"\bDe\s+er(\d{4})", r"December \1", s)
    return s

In [30]:
TRAILING_PAGE = re.compile(r"\s+\d{1,3}\s*$")
def strip_trailing_page(t): 
    return TRAILING_PAGE.sub("", t).strip()

In [92]:
len(doc)

64

In [93]:
markdown, blocks = build_document("data/raw/Human_Resource_Policy_Manual_PRAAN.2020.pdf", Path("data/processed"), max_pages = len(doc))

page   1/64  angle=+0.00  blocks+=8  2.6s
page   2/64  angle=-0.25  blocks+=36  3.3s
page   3/64  angle=-0.25  blocks+=76  3.8s
page   4/64  angle=+0.00  blocks+=115  5.2s
page   5/64  angle=-0.25  blocks+=154  5.2s
page   6/64  angle=-0.25  blocks+=193  6.0s
page   7/64  angle=-0.50  blocks+=230  4.7s
page   8/64  angle=-0.25  blocks+=267  8.8s
page   9/64  angle=-0.25  blocks+=310  11.1s
page  10/64  angle=-0.25  blocks+=350  10.0s
page  11/64  angle=-0.25  blocks+=403  10.5s
page  12/64  angle=-0.25  blocks+=431  6.3s
page  13/64  angle=-0.50  blocks+=468  7.1s
page  14/64  angle=-0.25  blocks+=517  6.8s
page  15/64  angle=+0.00  blocks+=565  9.8s
page  16/64  angle=-1.00  blocks+=611  10.9s
page  17/64  angle=-0.50  blocks+=656  10.9s
page  18/64  angle=-0.25  blocks+=703  10.8s
page  19/64  angle=-0.25  blocks+=741  9.9s
page  20/64  angle=+0.00  blocks+=773  8.6s
page  21/64  angle=-0.25  blocks+=826  11.7s
page  22/64  angle=-0.50  blocks+=873  11.1s
page  23/64  angle=-0.25  bl

In [94]:
print(markdown[ : 3000])

<!-- page 1 -->

preian

Human Resource

Policy Manual

Approved:December 28,2006

Updated:December 28,2020

Participatory Research Action Network-PRAAN

Email:pranbd.org|Phone:01919231722

www.pranbd.org

<!-- page 2 -->

Listof Acronyms

CV Curriculum Vitae

PM Program Manager

DPM Deputy Program Manager

PERDIEM Daily Subsistence Allowance

CE/CEO Chief Executive Officer

DD Deputy Director

GoB Governmentof Bangladesh

HRD Human Resource Development

HRIS Human Resource Information System

HRM Human Resource Management

JD Job Description

MEAL Monitoring,Evaluation,AccountabilityandLearning

MIS Management Information System

MF Manager-Finance

MP Manager-Program

NGO Non-Government Organization(Voluntary development.)

NGOAB NGOAffairs Bureau

PAMC Policy Advocacy and Mass Communication

PP Project Proposal

RMED Research,Monitoring,EvaluationandDocumentation

ToA Table of Authority

ToR Termsof Reference

Sharmind Neelormi

Nurut Alam Masud Chairperson, PRAN

Chief Executive.PR

In [95]:
print(markdown)

<!-- page 1 -->

preian

Human Resource

Policy Manual

Approved:December 28,2006

Updated:December 28,2020

Participatory Research Action Network-PRAAN

Email:pranbd.org|Phone:01919231722

www.pranbd.org

<!-- page 2 -->

Listof Acronyms

CV Curriculum Vitae

PM Program Manager

DPM Deputy Program Manager

PERDIEM Daily Subsistence Allowance

CE/CEO Chief Executive Officer

DD Deputy Director

GoB Governmentof Bangladesh

HRD Human Resource Development

HRIS Human Resource Information System

HRM Human Resource Management

JD Job Description

MEAL Monitoring,Evaluation,AccountabilityandLearning

MIS Management Information System

MF Manager-Finance

MP Manager-Program

NGO Non-Government Organization(Voluntary development.)

NGOAB NGOAffairs Bureau

PAMC Policy Advocacy and Mass Communication

PP Project Proposal

RMED Research,Monitoring,EvaluationandDocumentation

ToA Table of Authority

ToR Termsof Reference

Sharmind Neelormi

Nurut Alam Masud Chairperson, PRAN

Chief Executive.PR